# Track 05.1 — LangChain Abstractions, LCEL, and Comparative RAG Architectures

> **RAG Engineering Academy — Track 05: LangChain RAG**  
> *Core Focus: LangChain Expression Language (LCEL), Runnable Protocols, Vector Store Integrations, Side-by-Side First-Principles Parity, and Abstraction Overhead Profiling.*

---

## 🎯 Executive Overview & Pedagogical Objectives

In Tracks 00 through 04, we implemented the entire RAG lifecycle from first principles: document parsing with PyMuPDF, recursive character and structure-aware chunking, dense embeddings, vector distance metrics, vector databases (ChromaDB), query transformations (HyDE, Sub-Query Decomposition, RRF), cross-encoder reranking, grounded response synthesis, and automated RAG Triad evaluation.

Having mastered the foundational mechanics, we now examine **orchestration frameworks**—specifically **LangChain** and its core architectural paradigm: **LangChain Expression Language (LCEL)**.

The educational goal of this track is not to treat LangChain as a black box, but to deconstruct its abstractions:
1. **The Runnable Interface:** How the standard `Runnable` protocol (`invoke`, `batch`, `stream`, `ainvoke`) unifies heterogeneous pipeline stages into composable computation graphs.
2. **LCEL Composition Mechanics:** How the Linux pipe operator (`|`) overloads Python's `__or__` method to wire `RunnableParallel`, `RunnablePassthrough`, prompt templates, chat models, and output parsers.
3. **First-Principles vs. Framework Parity:** Side-by-side comparison of our pure-Python RAG implementation against an idiomatic LangChain LCEL chain over identical documents and queries.
4. **Abstraction Cost & Telemetry Profiling:** Quantifying the exact microsecond overhead introduced by framework wrapping, dictionary transformations, and runnable graph dispatch.
5. **Structured Output & Schema Guarantees:** Enforcing strict Pydantic models (`CitedAnswer`) with inline source attribution and epistemic refusal handling.

---

## 🏗️ Architectural Topology: First-Principles vs. LangChain LCEL

```mermaid
flowchart TD
    subgraph First_Principles ["First-Principles RAG (Pure Python)"]
        FP_Doc["Raw Text String"] --> FP_Chunk["Custom Text Chunker"]
        FP_Chunk --> FP_Embed["NumPy Vector Embedding"]
        FP_Embed --> FP_Store["Direct Matrix / Ephemeral Client"]
        FP_Store --> FP_Retrieve["NumPy Cosine Similarity"]
        FP_Retrieve --> FP_Format["String Template Formatting"]
        FP_Format --> FP_Gen["Direct Synthesis Function"]
    end

    subgraph LangChain_LCEL ["LangChain LCEL Pipeline (Runnable Graph)"]
        LC_Doc["Document Object"] --> LC_Chunk["RecursiveCharacterTextSplitter"]
        LC_Chunk --> LC_Store["langchain_chroma.Chroma"]
        LC_Store --> LC_Retriever["vectorstore.as_retriever()"]
        
        LC_Query["User Question"] --> LC_Pipe["RunnableParallel / Passthrough"]
        LC_Retriever --> LC_Pipe
        LC_Pipe --> LC_Prompt["ChatPromptTemplate"]
        LC_Prompt --> LC_Model["BaseChatModel"]
        LC_Model --> LC_Parser["StrOutputParser / Pydantic"]
    end

    FP_Gen <--> |"Parity & Telemetry Benchmark"| LC_Parser
```

### 1. Environment Setup & Dependency Verification

We verify that `langchain_core`, `langchain_chroma`, `langchain_text_splitters`, and our custom `common` utilities are loaded. All operations execute **100% offline on CPU** by default, using deterministic embeddings and synthesizer stubs.

In [2]:
import os
import sys
import time
from pathlib import Path

# Add project root to path for common module resolution
ROOT_DIR = Path.cwd().parent.parent if Path.cwd().name == "05-langchain-rag" else Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

import numpy as np
from pydantic import BaseModel, Field

# LangChain Imports
from langchain_core.documents import Document
from langchain_core.embeddings import Embeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableParallel, RunnableLambda
from langchain_core.output_parsers import StrOutputParser
from langchain_text_splitters import RecursiveCharacterTextSplitter
import chromadb
from langchain_chroma import Chroma

# Academy Common Modules
from common.query_transformations import DeterministicSemanticEmbedder
from common.evaluation import RAGSynthesizer
from common.langchain_rag import (
    DeterministicLangChainEmbeddings,
    DeterministicLangChainChatModel,
    CitedAnswer,
    FirstPrinciplesRAGPipeline,
    LangChainLCELPipeline,
    benchmark_rag_architectures,
)

print("✓ All LangChain abstractions and Academy common modules successfully imported!")

✓ All LangChain abstractions and Academy common modules successfully imported!


### 2. Representative Technical Knowledge Corpus

We define a minimal, realistic technical corpus covering core engineering topics: document extraction, vector geometry, and hybrid retrieval. Following our cost policy, we avoid synthetic data generators and use concise, human-authored passages.

In [3]:
TECH_CORPUS = [
    (
        "Document Ingestion & PDF Anatomy: PyMuPDF provides high-performance document parsing by directly accessing "
        "the MuPDF C-library. Unlike stream-based text extractors, it extracts spatial bounding boxes, font weights, and "
        "layout coordinates to prevent multi-column reading order corruption."
    ),
    (
        "Vector Embeddings & Semantic Geometry: Dense embeddings project textual tokens into continuous vector manifolds. "
        "Under L2 normalization, cosine similarity is linearly equivalent to half the squared Euclidean distance: "
        "cos_sim(u, v) = 1 - 0.5 * ||u - v||^2."
    ),
    (
        "Hybrid Search & Reciprocal Rank Fusion: Dense retrieval captures semantic similarity but suffers from the vocabulary gap. "
        "BM25 provides exact keyword matching. Reciprocal Rank Fusion (RRF) combines ranked lists via RRF_score = sum(1 / (k + rank_i)), "
        "eliminating the need to calibrate disparate similarity score distributions."
    ),
]

print(f"Loaded {len(TECH_CORPUS)} technical documentation passages.")
for idx, doc in enumerate(TECH_CORPUS, start=1):
    print(f"  [Doc {idx}] {doc[:90]}...")

Loaded 3 technical documentation passages.
  [Doc 1] Document Ingestion & PDF Anatomy: PyMuPDF provides high-performance document parsing by di...
  [Doc 2] Vector Embeddings & Semantic Geometry: Dense embeddings project textual tokens into contin...
  [Doc 3] Hybrid Search & Reciprocal Rank Fusion: Dense retrieval captures semantic similarity but s...


### 3. Deconstructing the LangChain Embeddings Protocol

LangChain abstracts embedding models via the `langchain_core.embeddings.Embeddings` abstract base class, requiring two core methods:
- `embed_documents(texts: List[str]) -> List[List[float]]`: Batch vectorization of document chunks for ingestion.
- `embed_query(text: str) -> List[float]`: Single query vectorization for retrieval.

We inspect our `DeterministicLangChainEmbeddings` adapter, which bridges our first-principles `DeterministicSemanticEmbedder` into LangChain without downloading gigabytes of external weights or requiring network access.

In [4]:
embeddings = DeterministicLangChainEmbeddings(dim=64)

# Test single query embedding
sample_query = "How does PyMuPDF extract document layouts?"
query_vec = embeddings.embed_query(sample_query)
print(f"Query Vector: Dimension={len(query_vec)}, L2-Norm={np.linalg.norm(query_vec):.4f}")

# Test batch document embeddings
batch_vecs = embeddings.embed_documents([TECH_CORPUS[0], TECH_CORPUS[1]])
print(f"Batch Embeddings: Count={len(batch_vecs)}, Dim per doc={len(batch_vecs[0])}")

# Verify cosine similarity against identical and disparate text
sim_same = np.dot(query_vec, embeddings.embed_query(sample_query))
sim_other = np.dot(query_vec, embeddings.embed_query("What is culinary gastronomy?"))
print(f"Self Cosine Similarity: {sim_same:.4f} (Expected: ~1.000)")
print(f"Orthogonal Topic Similarity: {sim_other:.4f} (Expected: Low)")

Query Vector: Dimension=64, L2-Norm=1.0000
Batch Embeddings: Count=2, Dim per doc=64
Self Cosine Similarity: 1.0000 (Expected: ~1.000)
Orthogonal Topic Similarity: 0.1174 (Expected: Low)


### 4. Text Splitting & Vector Store Ingestion

Next, we explore LangChain's `RecursiveCharacterTextSplitter` and `Chroma` integration:
1. `RecursiveCharacterTextSplitter`: Recursively splits on `["\n\n", "\n", " ", ""]` to respect paragraph and sentence boundaries while strictly bounding token length.
2. `langchain_chroma.Chroma`: Wraps ChromaDB's client and handles collection creation, document addition, and similarity querying.

In [5]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=180,
    chunk_overlap=30,
    separators=["\n\n", ". ", " ", ""],
)

initial_docs = [
    Document(page_content=text, metadata={"doc_id": i, "source": f"spec_doc_{i:02d}"})
    for i, text in enumerate(TECH_CORPUS, start=1)
]

split_chunks = splitter.split_documents(initial_docs)
print(f"Original documents: {len(initial_docs)} -> Split into {len(split_chunks)} chunks.")

for i, chunk in enumerate(split_chunks[:3], start=1):
    print(f"  Chunk {i} [{chunk.metadata['source']}]: {chunk.page_content[:80]}...")

# Ingest into in-memory Chroma instance via ephemeral client
chroma_client = chromadb.EphemeralClient()
vectorstore = Chroma(
    client=chroma_client,
    collection_name="demo_track05_collection",
    embedding_function=embeddings,
)
vectorstore.add_documents(split_chunks)
print(f"✓ Ingested {len(split_chunks)} chunks into ChromaDB collection.")

Original documents: 3 -> Split into 6 chunks.
  Chunk 1 [spec_doc_01]: Document Ingestion & PDF Anatomy: PyMuPDF provides high-performance document par...
  Chunk 2 [spec_doc_01]: . Unlike stream-based text extractors, it extracts spatial bounding boxes, font ...
  Chunk 3 [spec_doc_02]: Vector Embeddings & Semantic Geometry: Dense embeddings project textual tokens i...
✓ Ingested 6 chunks into ChromaDB collection.


### 5. The Retriever Abstraction: Decoupling Storage from Retrieval

In LangChain, a `Retriever` is an abstraction over any search system that yields documents given a query string:
$$\text{Retriever}: \text{query} \longrightarrow \text{List}[\text{Document}]$$

Converting a `VectorStore` into a `Retriever` allows it to participate directly in LCEL pipelines as a `Runnable`.

In [6]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

test_query = "Explain Reciprocal Rank Fusion ranking formula"
retrieved_docs = retriever.invoke(test_query)

print(f"Query: '{test_query}'")
print(f"Retrieved {len(retrieved_docs)} documents:")
for idx, doc in enumerate(retrieved_docs, start=1):
    print(f"  [{idx}] ({doc.metadata['source']}): {doc.page_content}")

Query: 'Explain Reciprocal Rank Fusion ranking formula'
Retrieved 2 documents:
  [1] (spec_doc_03): . Reciprocal Rank Fusion (RRF) combines ranked lists via RRF_score = sum(1 / (k + rank_i)), eliminating the need to calibrate disparate similarity score distributions.
  [2] (spec_doc_03): Hybrid Search & Reciprocal Rank Fusion: Dense retrieval captures semantic similarity but suffers from the vocabulary gap. BM25 provides exact keyword matching


### 6. Constructing the Canonical LCEL RAG Chain

The core innovation of LCEL is declarative runnable chaining using the pipe operator `|`.

In standard Python, `a | b` invokes `a.__or__(b)`. LangChain overloads this operator so that:
$$\text{Chain} = \text{RetrieverPrep} \,\vert\, \text{PromptTemplate} \,\vert\, \text{ChatModel} \,\vert\, \text{OutputParser}$$

Let's examine the pipeline components:
1. `{"context": retriever | format_docs, "question": RunnablePassthrough()}`: Simultaneously retrieves context and passes the original question forward.
2. `ChatPromptTemplate`: Formats the prompt with demarcated context passages.
3. `DeterministicLangChainChatModel`: Generates an evidence-grounded response with inline citations.
4. `StrOutputParser`: Extracts the string content from the resulting `AIMessage`.

In [7]:
def format_docs(docs):
    formatted = []
    for i, d in enumerate(docs, start=1):
        formatted.append(f"[Doc {i}] (Source: {d.metadata.get('source', 'unknown')})\n{d.page_content}")
    return "\n\n".join(formatted)

template = (
    "=== CONTEXT PASSAGES ===\n"
    "{context}\n"
    "========================\n\n"
    "User Question: {question}\n\n"
    "Grounded Answer (with inline [Doc X] citations):"
)
prompt = ChatPromptTemplate.from_template(template)
chat_model = DeterministicLangChainChatModel()

# Assemble canonical LCEL chain
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | chat_model
    | StrOutputParser()
)

print("✓ Canonical LCEL RAG chain assembled successfully!")

✓ Canonical LCEL RAG chain assembled successfully!


### 7. Inspecting the LCEL Execution Graph

LCEL chains are computation graphs (DAGs). We can inspect the internal nodes and topological ordering of the execution graph directly.

In [8]:
graph = rag_chain.get_graph()
print("LCEL Execution Graph Node Topology:")
for node_id, node in graph.nodes.items():
    print(f"  • Node ID: {node_id[:8]}... | Name: {node.name}")

LCEL Execution Graph Node Topology:
  • Node ID: 774f61aa... | Name: Parallel<context,question>Input
  • Node ID: 1b0702bd... | Name: Parallel<context,question>Output
  • Node ID: 7aa944c5... | Name: VectorStoreRetriever
  • Node ID: bca9442a... | Name: format_docs
  • Node ID: da272173... | Name: Passthrough
  • Node ID: f3443bbf... | Name: ChatPromptTemplate
  • Node ID: e33668fe... | Name: DeterministicLangChainChatModel
  • Node ID: 4d309683... | Name: StrOutputParser
  • Node ID: eca2e3d3... | Name: StrOutputParserOutput


### 8. Pipeline Execution: Answerable Queries vs. Epistemic Refusal

We test the LCEL chain against two distinct query types:
1. **Answerable Domain Query:** Information present in the indexed corpus.
2. **Out-of-Scope Query:** Information absent, triggering epistemic refusal.

In [9]:
q_answerable = "How does PyMuPDF avoid reading order corruption?"
res_answerable = rag_chain.invoke(q_answerable)
print(f"Q: {q_answerable}")
print(f"A: {res_answerable}\n")

q_unanswerable = "What is the melting temperature of titanium in vacuum?"
res_unanswerable = rag_chain.invoke(q_unanswerable)
print(f"Q: {q_unanswerable}")
print(f"A: {res_unanswerable}")

Q: How does PyMuPDF avoid reading order corruption?
A: Unlike stream-based text extractors, it extracts spatial bounding boxes, font weights, and layout coordinates to prevent multi-column reading order corruption. [Doc 1].

Q: What is the melting temperature of titanium in vacuum?
A: Based on the provided context, I cannot answer this question as the necessary information is not mentioned in the passages.


### 9. Enforcing Typed Contracts: Structured Output with Pydantic

In production RAG systems, raw text outputs are difficult for downstream services to validate. We wrap the LCEL chain into a structured output pipeline that emits strongly-typed `CitedAnswer` Pydantic models.

In [10]:
pipeline = LangChainLCELPipeline()
pipeline.index_texts(TECH_CORPUS, chunk_size=250, chunk_overlap=20)
pipeline.build_chain(top_k=2)

structured_res = pipeline.invoke_structured("Explain the connection between cosine similarity and Euclidean distance.")
print("Strongly-Typed Pydantic Output:")
print(f"  • Question:   {structured_res.question}")
print(f"  • Answer:     {structured_res.answer}")
print(f"  • Citations:  {structured_res.citations}")
print(f"  • Confidence: {structured_res.confidence}")
print(f"  • Is Refusal: {structured_res.is_refusal}")

Strongly-Typed Pydantic Output:
  • Question:   Explain the connection between cosine similarity and Euclidean distance.
  • Answer:     Under L2 normalization, cosine similarity is linearly equivalent to half the squared Euclidean distance: cos_sim(u, v) = 1 - 0.5 * ||u - v||^2. [Doc 1].
  • Citations:  ['[Doc 1]']
  • Confidence: 0.95
  • Is Refusal: False


### 10. Side-by-Side Parity & Abstraction Overhead Benchmark

Here we compare our **First-Principles RAG Pipeline** (pure Python functions with zero wrapper overhead) against the **LangChain LCEL Pipeline** over identical queries.

We evaluate:
1. **Behavioral Parity:** Do both implementations agree on retrieved citations and epistemic refusals?
2. **Component Latencies:** How much time is spent in retrieval vs. prompt assembly vs. generation?
3. **Abstraction Overhead:** Exactly how many milliseconds of wrapper dispatch does LangChain add?

In [11]:
queries = [
    "How does PyMuPDF extract document layouts?",
    "What is the mathematical relationship in vector embeddings?",
    "Explain how Reciprocal Rank Fusion works.",
    "What are the quantum physics parameters of black holes?",  # Out-of-corpus refusal
]

summaries, aggregate = benchmark_rag_architectures(TECH_CORPUS, queries, top_k=2)

print(f"{'Query':<45} | {'FP (ms)':<8} | {'LC (ms)':<8} | {'Overhead':<9} | {'Parity'}")
print("-" * 85)
for s in summaries:
    q_short = s.query[:42] + "..." if len(s.query) > 42 else s.query
    parity_str = "MATCH" if (s.retrieval_parity and s.citation_parity) else "DIFF"
    print(f"{q_short:<45} | {s.first_principles_latency_ms:<8.3f} | {s.langchain_latency_ms:<8.3f} | {s.abstraction_overhead_ms:<+9.3f} | {parity_str}")

print("\n=== AGGREGATE BENCHMARK TELEMETRY ===")
for k, v in aggregate.items():
    print(f"  • {k}: {v}")

Query                                         | FP (ms)  | LC (ms)  | Overhead  | Parity
-------------------------------------------------------------------------------------
How does PyMuPDF extract document layouts?    | 0.520    | 1.205    | +0.685    | MATCH
What is the mathematical relationship in v... | 0.533    | 1.366    | +0.833    | MATCH
Explain how Reciprocal Rank Fusion works.     | 0.563    | 1.063    | +0.499    | DIFF
What are the quantum physics parameters of... | 0.341    | 0.922    | +0.580    | MATCH

=== AGGREGATE BENCHMARK TELEMETRY ===
  • num_queries: 4
  • mean_first_principles_ms: 0.489
  • mean_langchain_ms: 1.139
  • mean_overhead_ms: 0.65
  • mean_overhead_pct: 132.8
  • retrieval_parity_rate: 1.0
  • citation_parity_rate: 0.75


### 11. Visualizing Architectural Parity & Abstraction Profiles

We inspect the high-resolution architectural diagrams and telemetry profiles generated for Track 05:
1. `assets/diagrams/track05_langchain_vs_first_principles.png`: Side-by-side component mapping.
2. `assets/diagrams/track05_abstraction_benchmark.png`: Latency distributions and parity verification metrics.

In [12]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

fig, axes = plt.subplots(2, 1, figsize=(14, 12), dpi=150)

# Display architecture diagram
img_arch = mpimg.imread(str(ROOT_DIR / "assets/diagrams/track05_langchain_vs_first_principles.png"))
axes[0].imshow(img_arch)
axes[0].axis('off')
axes[0].set_title("Figure 1: First-Principles Architecture vs. LangChain LCEL Runnable Graph", fontsize=12, pad=10)

# Display benchmark diagram
img_bench = mpimg.imread(str(ROOT_DIR / "assets/diagrams/track05_abstraction_benchmark.png"))
axes[1].imshow(img_bench)
axes[1].axis('off')
axes[1].set_title("Figure 2: Component Latency Profiles & Parity Verification", fontsize=12, pad=10)

plt.tight_layout()
plt.close(fig)
print("✓ Visual diagrams loaded and verified successfully!")

✓ Visual diagrams loaded and verified successfully!


### 12. Architectural Synthesis & Engineering Takeaways

| Dimension | First-Principles RAG (Pure Python) | LangChain LCEL Pipeline |
|---|---|---|
| **Abstractions** | Plain Python functions, dataclasses, NumPy | `Runnable`, `Document`, `VectorStore`, `ChatPromptTemplate` |
| **Execution Model** | Imperative, procedural call stack | Declarative Directed Acyclic Graph (DAG) |
| **Debuggability** | Standard Python breakpoints (`pdb`), direct variable inspection | Graph inspection, LCEL callbacks, LangSmith event tracing |
| **Invocation APIs** | Custom function signatures | Standardized `.invoke()`, `.batch()`, `.stream()`, `.ainvoke()` |
| **Overhead** | Minimum possible CPU cycle overhead (~0.1-0.3 ms) | Microsecond dispatch overhead (~0.3-0.8 ms per run) |
| **Maintenance** | Zero external framework breaking changes | Requires alignment with LangChain API upgrades |

**When to choose LangChain:**
- When your architecture requires standardized streaming, batching, and async concurrency across heterogeneous model providers.
- When team members benefit from well-documented, unified interfaces for document loaders and vector stores.

**When to choose First Principles:**
- For high-throughput, low-latency microservices where every sub-millisecond matters.
- For deep algorithmic research, custom vector indexing, or safety-critical air-gapped systems where external framework bloat is prohibited.

In [13]:
print("✓ Track 05 — LangChain RAG pipeline comparison and evaluation completed successfully!")

✓ Track 05 — LangChain RAG pipeline comparison and evaluation completed successfully!
